# Part A: linear models on sparse features

Predict a 1–5 rating from review text on Wildberries (WB) and Amazon. Classical ML only.

**Protocol.** Fit on 80% of `train`, score on the other 20% (split by product, seed 0). Main metric: macro-F1, second: MAE. The test split is used once, in the last section, for the final text-only model.

Finished runs are stored in `results.csv` and skipped. To retrain an experiment, delete its row. Run from this folder.

In [1]:
import functools
import itertools
import json
import pathlib
import re
import time

import numpy as np
import optuna
import pandas as pd
import pymorphy3
import scipy.sparse as sp
from gensim import downloader
from gensim import models as gensim_models
from nltk.stem import snowball
from sklearn import base
from sklearn import calibration
from sklearn import feature_selection
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection
from sklearn import naive_bayes
from sklearn import preprocessing
from sklearn import svm
from sklearn.feature_extraction import text as sk_text

SPLITS = pathlib.Path('../../data/splits')
RESULTS = pathlib.Path('results.csv')
DATASETS = ['wb', 'amazon']
SEED = 0
VAL_SIZE = 0.2
INNER_SIZE = 0.25
CLASSES = np.arange(1, 6)
EMOJI = re.compile('[\U0001F300-\U0001FAFF☀-➿]')
LANG = {'wb': 'russian', 'amazon': 'english'}
CATEGORY = {'wb': 'subject_root', 'amazon': 'category'}
BIAS_GRID = np.linspace(-3, 3, 61)
WEIGHT_POWER = {'balanced': 1, 'sqrt': 0.5, 'none': 0, 'under': 0, 'under5': 0,
                'over': 0}
BASE_FEATURES = {'text': 'full', 'word': [1, 2], 'char': None, 'min_df': 3,
                 'max_features': 200_000, 'char_max_features': 300_000,
                 'sublinear': True, 'binary': False, 'meta': [],
                 'select_k': None, 'vectors': None, 'vector_scale': 1.0}
BASE_MODEL = {'name': 'logreg', 'C': 2.0}
ALL_META = ['flags', 'length', 'punct', 'category']
BEST = {'from': 'best'}
VECTOR_MODELS = {'wb': ('word2vec-ruscorpora-300', None),
                 'amazon': ('word2vec-google-news-300', 1_000_000)}

/Users/nick/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Data:
    """Train split of one platform, divided by product into fit and val."""

    def __init__(self, name):
        train, test = [pd.read_csv(SPLITS / f'{name}_{part}.csv.gz',
                                   keep_default_na=False)
                       for part in ['train', 'test']]
        # Test rows go last; they are only read by the final test section.
        df = pd.concat([train, test], ignore_index=True)
        df['rating'] = df.rating.astype(int)
        df['n_words'] = df.full_text.str.split().str.len()
        splitter = model_selection.GroupShuffleSplit(
            n_splits=1, test_size=VAL_SIZE, random_state=SEED)
        self.fit_idx, self.val_idx = next(
            splitter.split(train, groups=train.product_id))
        self.df = df
        self.y_fit = df.rating.to_numpy()[self.fit_idx]
        self.y_val = df.rating.to_numpy()[self.val_idx]
        self.groups_fit = df.product_id.to_numpy()[self.fit_idx]
        inner = model_selection.GroupShuffleSplit(
            n_splits=1, test_size=INNER_SIZE, random_state=SEED)
        tr, te = next(inner.split(self.fit_idx, groups=self.groups_fit))
        self.splits = {'val': (self.fit_idx, self.val_idx),
                       'inner': (self.fit_idx[tr], self.fit_idx[te]),
                       'test': (np.arange(len(train)),
                                np.arange(len(train), len(df)))}


def normalize(s):
    """Lowercases, maps emoji and '!' to tokens, collapses digits."""
    s = s.lower().replace('ё', 'е')
    s = EMOJI.sub(lambda m: f' emoji{ord(m.group()):x} ', s)
    s = re.sub(r'!+', ' excl ', s)
    return re.sub(r'\d+', '0', s)


def map_words(texts, fn):
    """Applies fn to every word, computing it once per unique word."""
    tokens = texts.str.findall(r'\w+')
    vocab = {w: fn(w) for w in set(w for ts in tokens for w in ts)}
    return tokens.map(lambda ts: ' '.join(vocab[w] for w in ts))


@functools.cache
def make_text(dataset, variant):
    """Builds one text variant: full, plain, clean, stem or lemma."""
    df = DATA[dataset].df
    if variant == 'clean':
        return df.full_text.map(normalize)
    if variant == 'stem':
        stem = snowball.SnowballStemmer(LANG[dataset]).stem
        return map_words(make_text(dataset, 'clean'), stem)
    if variant == 'lemma':
        morph = pymorphy3.MorphAnalyzer()
        return map_words(make_text(dataset, 'clean'),
                         lambda w: morph.parse(w)[0].normal_form)
    if variant == 'plain' and dataset == 'wb':
        return (df.text + ' ' + df.pros + ' ' + df.cons).str.strip()
    # Amazon has no field markers, so its plain text equals the full text.
    return df.full_text


DATA = {name: Data(name) for name in DATASETS}
def describe(d):
    """Returns split sizes and the rating shares of train."""
    train_idx, test_idx = d.splits['test']
    shares = d.df.rating.iloc[train_idx].value_counts(normalize=True)
    return {'fit': len(d.fit_idx), 'val': len(d.val_idx),
            'test': len(test_idx), **shares.round(3).sort_index()}


pd.DataFrame({name: describe(d) for name, d in DATA.items()})

,wb,amazon
fit,191457.000,192532.000
val,50138.000,46766.000
test,58127.000,60736.000
1,0.071,0.117
2,0.028,0.051
3,0.059,0.069
4,0.082,0.107
5,0.759,0.655


In [3]:
@functools.cache
def vector_text(dataset):
    """Returns the text matched against the pretrained vocabulary."""
    variant = 'lemma' if dataset == 'wb' else 'clean'
    return make_text(dataset, variant).str.replace('ё', 'е')


@functools.cache
def load_vectors(dataset):
    """Returns {word: row} and unit vectors of the words seen in train."""
    name, limit = VECTOR_MODELS[dataset]
    model = gensim_models.KeyedVectors.load_word2vec_format(
        downloader.load(name, return_path=True), binary=True, limit=limit)
    # RusCorpora keys are lemma_POS; the most frequent POS of a lemma wins.
    index = {}
    for key, row in model.key_to_index.items():
        word = key.rsplit('_', 1)[0].replace('ё', 'е')
        index.setdefault(word if dataset == 'wb' else key, row)
    tokens = vector_text(dataset).str.findall(r'\w+')
    seen = set(itertools.chain.from_iterable(tokens))
    vocab = {w: i for i, w in enumerate(sorted(seen & index.keys()))}
    rows = [index[w] for w in vocab]
    return vocab, preprocessing.normalize(model.vectors[rows])


def pool_vectors(dataset, mode, part):
    """Returns L2-normalized count- or IDF-weighted mean word vectors."""
    vocab, vectors = load_vectors(dataset)
    texts = vector_text(dataset)
    tr_idx, te_idx = DATA[dataset].splits[part]
    weights = sk_text.TfidfVectorizer(
        vocabulary=vocab, token_pattern=r'\w+', lowercase=False,
        use_idf=mode == 'idf', norm=None, dtype=np.float32)
    x_fit = weights.fit_transform(texts.iloc[tr_idx]) @ vectors
    x_val = weights.transform(texts.iloc[te_idx]) @ vectors
    return preprocessing.normalize(x_fit), preprocessing.normalize(x_val)

In [4]:
def meta_columns(df, dataset, kind):
    """Non-negative dense meta features of one kind."""
    if kind == 'flags' and dataset == 'wb':
        return df[['has_text', 'has_pros', 'has_cons']].to_numpy(float)
    if kind == 'flags':
        return (df.title != '').to_numpy(float)[:, None]
    if kind == 'length':
        return np.log1p(np.c_[df.n_words, df.full_text.str.len()])
    text = df.full_text.str
    caps = text.count('[A-ZА-ЯЁ]') / text.len().clip(lower=1)
    return np.c_[np.log1p(text.count('!')), np.log1p(text.count(r'\?')),
                 np.log1p(text.count(EMOJI)), caps]


@functools.lru_cache(maxsize=1)
def build_features(dataset, spec_json, part='val'):
    """Fits the feature pipeline on one split, returns (x_fit, x_val)."""
    spec = {**BASE_FEATURES, **json.loads(spec_json)}
    data = DATA[dataset]
    texts = make_text(dataset, spec['text'])
    tr_idx, te_idx = data.splits[part]
    fit_texts, val_texts = texts.iloc[tr_idx], texts.iloc[te_idx]
    df_fit, df_val = data.df.iloc[tr_idx], data.df.iloc[te_idx]
    common = dict(min_df=spec['min_df'], sublinear_tf=spec['sublinear'],
                  binary=spec['binary'], dtype=np.float32)
    vectorizers = []
    if spec['word']:
        vectorizers.append(sk_text.TfidfVectorizer(
            ngram_range=tuple(spec['word']),
            max_features=spec['max_features'], **common))
    if spec['char']:
        vectorizers.append(sk_text.TfidfVectorizer(
            analyzer='char_wb', ngram_range=tuple(spec['char']),
            max_features=spec['char_max_features'], **common))
    blocks_fit = [v.fit_transform(fit_texts) for v in vectorizers]
    blocks_val = [v.transform(val_texts) for v in vectorizers]
    if spec['vectors']:
        v_fit, v_val = pool_vectors(dataset, spec['vectors'], part)
        blocks_fit.append(
            sp.csr_matrix(v_fit * spec['vector_scale'], dtype=np.float32))
        blocks_val.append(
            sp.csr_matrix(v_val * spec['vector_scale'], dtype=np.float32))
    for kind in spec['meta']:
        if kind == 'category':
            column = [CATEGORY[dataset]]
            encoder = preprocessing.OneHotEncoder(
                handle_unknown='ignore', dtype=np.float32)
            blocks_fit.append(encoder.fit_transform(df_fit[column]))
            blocks_val.append(encoder.transform(df_val[column]))
            continue
        m_fit = meta_columns(df_fit, dataset, kind)
        m_val = meta_columns(df_val, dataset, kind)
        scale = np.maximum(m_fit.max(axis=0), 1e-9)
        blocks_fit.append(sp.csr_matrix(m_fit / scale, dtype=np.float32))
        blocks_val.append(
            sp.csr_matrix(np.clip(m_val / scale, 0, None), dtype=np.float32))
    x_fit, x_val = sp.hstack(blocks_fit).tocsr(), sp.hstack(blocks_val).tocsr()
    if spec['select_k']:
        selector = feature_selection.SelectKBest(
            feature_selection.chi2, k=spec['select_k'])
        selector.fit(x_fit, data.df.rating.to_numpy()[tr_idx])
        x_fit, x_val = selector.transform(x_fit), selector.transform(x_val)
    return x_fit, x_val

In [5]:
class NBLogReg(base.BaseEstimator):
    """One-vs-rest logistic regressions on NB log-count-ratio features."""

    def __init__(self, C=1.0, alpha=1.0):
        self.C = C
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        """Fits one binary model per class."""
        self.classes_ = np.unique(y)
        self.ratios_, self.models_ = [], []
        for c in self.classes_:
            p = self.alpha + np.asarray(x[y == c].sum(axis=0)).ravel()
            q = self.alpha + np.asarray(x[y != c].sum(axis=0)).ravel()
            r = np.log((p / p.sum()) / (q / q.sum())).astype(np.float32)
            model = linear_model.LogisticRegression(C=self.C, max_iter=1000)
            model.fit(x.multiply(r).tocsr(), (y == c).astype(int),
                      sample_weight=sample_weight)
            self.ratios_.append(r)
            self.models_.append(model)
        return self

    def decision_function(self, x):
        """Returns per-class margins."""
        return np.column_stack([
            m.decision_function(x.multiply(r).tocsr())
            for m, r in zip(self.models_, self.ratios_)])

    def predict(self, x):
        """Returns the class with the largest margin."""
        return self.classes_[self.decision_function(x).argmax(axis=1)]


MODELS = {
    'logreg': lambda **p: linear_model.LogisticRegression(max_iter=1000, **p),
    'linsvc': svm.LinearSVC,
    # Linear SVM with margins mapped to probabilities on 3 internal folds.
    'linsvc_cal': lambda C, method: calibration.CalibratedClassifierCV(
        svm.LinearSVC(C=C), method=method, cv=3),
    'sgd': lambda **p: linear_model.SGDClassifier(random_state=SEED, **p),
    'ridge_clf': linear_model.RidgeClassifier,
    'cnb': naive_bayes.ComplementNB,
    'mnb': naive_bayes.MultinomialNB,
    'nbsvm': NBLogReg,
    'pa': lambda C: linear_model.SGDClassifier(
        loss='hinge', penalty=None, learning_rate='pa1', eta0=C,
        random_state=SEED),
    'ridge_reg': linear_model.Ridge,
}


def make_model(spec):
    """Creates an unfitted model from {'name': ..., **params}."""
    params = {k: v for k, v in spec.items() if k != 'name'}
    return MODELS[spec['name']](**params)


FILTERS = {
    'drop_short': lambda df: df.n_words.to_numpy() >= 3,
    'dedup': lambda df: ~df.full_text.duplicated().to_numpy(),
    'drop_not_product': lambda df: ~df.excluded_from_rating.str.contains(
        'notProduct').to_numpy(),
}


def class_weights(y, power):
    """Returns balanced class weights raised to a power."""
    counts = pd.Series(y).value_counts()
    return ((len(y) / (len(CLASSES) * counts)).reindex(y).to_numpy())**power


def select_rows(data, cfg):
    """Applies train-only filters and imbalance mode, returns rows, weights."""
    df = data.df.iloc[data.fit_idx]
    keep = np.ones(len(df), bool)
    for name in cfg['filters']:
        keep &= FILTERS[name](df)
    rows = np.flatnonzero(keep)
    y = data.y_fit[rows]
    rng = np.random.default_rng(SEED)
    counts = pd.Series(y).value_counts()
    mode = cfg['imbalance']
    if mode == 'under':
        rows = np.concatenate([
            rng.choice(rows[y == c], counts.min(), replace=False)
            for c in CLASSES])
    if mode == 'under5':
        # Only 5★ is cut down, to twice the next largest class.
        cap = min(2 * counts.drop(5).max(), counts[5])
        rows = np.concatenate(
            [rows[y != 5], rng.choice(rows[y == 5], cap, replace=False)])
    if mode == 'over':
        # Minority classes are duplicated up to half of the largest class.
        target = counts.max() // 2
        rows = np.concatenate([rows] + [
            rng.choice(rows[y == c], target - counts[c])
            for c in CLASSES if counts[c] < target])
    # A float imbalance mode is a direct power of the balanced weights.
    power = mode if isinstance(mode, float) else WEIGHT_POWER[mode]
    return rows, class_weights(data.y_fit[rows], power)

In [6]:
def scores(model, x):
    """Continuous output for decoders: log-proba, margins or regression."""
    if hasattr(model, 'predict_proba'):
        return np.log(np.clip(model.predict_proba(x), 1e-12, None))
    if hasattr(model, 'decision_function'):
        return model.decision_function(x)
    return model.predict(x)


def median_decode(log_proba):
    """Returns the median of the predicted distribution (MAE-optimal)."""
    cdf = np.cumsum(np.exp(log_proba), axis=1)
    return CLASSES[(cdf >= 0.5).argmax(axis=1)]


def tune_bias(s, y):
    """Finds per-class score offsets maximizing macro-F1."""
    def macro_f1(offsets):
        pred = CLASSES[(s + offsets).argmax(1)]
        return metrics.f1_score(y, pred, average='macro')

    bias = np.zeros(s.shape[1])
    for _, k in itertools.product(range(3), range(s.shape[1])):
        bias[k] = max(BIAS_GRID, key=lambda b: macro_f1(
            np.r_[bias[:k], b, bias[k + 1:]]))
    return bias


def tune_thresholds(s, y):
    """Finds regression cut points maximizing macro-F1."""
    def macro_f1(cuts):
        pred = CLASSES[np.digitize(s, cuts)]
        return metrics.f1_score(y, pred, average='macro')

    cuts = np.array([1.5, 2.5, 3.5, 4.5])
    for _, k in itertools.product(range(3), range(4)):
        lo = cuts[k - 1] + 0.01 if k else s.min()
        hi = cuts[k + 1] - 0.01 if k < 3 else s.max()
        cuts[k] = max(np.linspace(lo, hi, 50), key=lambda t: macro_f1(
            np.r_[cuts[:k], t, cuts[k + 1:]]))
    return cuts


def oof_scores(model, x, y, weight, groups):
    """Returns out-of-fold scores on the fit part, 3 folds by product."""
    splitter = model_selection.GroupKFold(n_splits=3)
    folds = list(splitter.split(x, y, groups))
    parts = [
        scores(base.clone(model).fit(x[tr], y[tr], sample_weight=weight[tr]),
               x[te]) for tr, te in folds]
    order = np.concatenate([te for _, te in folds])
    return np.concatenate(parts)[np.argsort(order)]


def predict(cfg, x_tr, y_tr, weight, groups, x_val):
    """Fits the model and decodes validation predictions."""
    model, decode = make_model(cfg['model']), cfg['decode']
    is_regression = cfg['model']['name'] == 'ridge_reg'
    if decode == 'bias':
        offsets = tune_bias(oof_scores(model, x_tr, y_tr, weight, groups), y_tr)
    if decode == 'thresholds':
        cuts = tune_thresholds(
            oof_scores(model, x_tr, y_tr, weight, groups), y_tr)
    model.fit(x_tr, y_tr, sample_weight=weight)
    if decode == 'argmax' and is_regression:
        return np.clip(np.rint(model.predict(x_val)), 1, 5).astype(int)
    if decode == 'argmax':
        return model.predict(x_val)
    s = scores(model, x_val)
    if decode == 'median':
        return median_decode(s)
    if decode == 'bias':
        return CLASSES[(s + offsets).argmax(axis=1)]
    return CLASSES[np.digitize(s, cuts)]


def evaluate(y, pred):
    """Returns macro-F1, MAE, accuracy and per-class F1."""
    per_class = metrics.f1_score(
        y, pred, labels=CLASSES, average=None, zero_division=0)
    return {'macro_f1': metrics.f1_score(y, pred, average='macro'),
            'mae': metrics.mean_absolute_error(y, pred),
            'accuracy': metrics.accuracy_score(y, pred),
            **{f'f1_{c}': v for c, v in zip(CLASSES, per_class)}}

In [7]:
def feat(**changes):
    """Returns the base feature spec with some fields changed."""
    return {**BASE_FEATURES, **changes}


def exp(exp_id, features=BEST, model=BASE_MODEL, imbalance='balanced',
        filters=(), decode='argmax', wb_only=False):
    """Returns one experiment config."""
    return dict(exp_id=exp_id, features=features, model=model,
                imbalance=imbalance, filters=filters, decode=decode,
                wb_only=wb_only)


def sweep(name, key, values, **fixed):
    """Returns one experiment per value of a model parameter."""
    params = [{**fixed, key: v} for v in values]
    return [exp(f'm_{name}_' + '_'.join(f'{k}{v}' for k, v in p.items()),
                model={'name': name, **p}) for p in params]


def is_text_only(spec_json):
    """Tells whether a stored feature spec has no meta features."""
    return not {**BASE_FEATURES, **json.loads(spec_json)}['meta']


def resolve_features(results, dataset, spec):
    """Replaces {'from': 'best' or 'best_text'} with a features group spec."""
    if 'from' not in spec:
        return spec
    rows = results[(results.dataset == dataset) & (results.group == 'features')]
    if spec['from'] == 'best_text':
        rows = rows[rows.features.map(is_text_only)]
    best = json.loads(rows.loc[rows.macro_f1.idxmax(), 'features'])
    return {**best, **{k: v for k, v in spec.items() if k != 'from'}}


def table(group):
    """Returns validation metrics of one group, best WB macro-F1 first."""
    results = pd.read_csv(RESULTS)
    wide = results[results.group == group].pivot(
        index='exp_id', columns='dataset',
        values=['macro_f1', 'mae', 'fit_sec'])
    return wide.sort_values(('macro_f1', 'wb'), ascending=False).round(4)


def run(group, experiments, datasets=tuple(DATASETS)):
    """Fits unlogged experiments on the datasets, returns the group table."""
    for dataset, cfg in itertools.product(datasets, experiments):
        results = pd.read_csv(RESULTS, float_precision='round_trip')
        done = (results.dataset == dataset) & (results.exp_id == cfg['exp_id'])
        if done.any() or (cfg['wb_only'] and dataset != 'wb'):
            continue
        data = DATA[dataset]
        spec = resolve_features(results, dataset, cfg['features'])
        x_fit, x_val = build_features(dataset, json.dumps(spec, sort_keys=True))
        rows, weight = select_rows(data, cfg)
        start = time.time()
        pred = predict(cfg, x_fit[rows], data.y_fit[rows], weight,
                       data.groups_fit[rows], x_val)
        row = {'dataset': dataset, 'group': group, 'exp_id': cfg['exp_id'],
               **evaluate(data.y_val, pred),
               'features': json.dumps(spec, ensure_ascii=False),
               'model': json.dumps(cfg['model'], ensure_ascii=False),
               'imbalance': cfg['imbalance'],
               'filters': json.dumps(list(cfg['filters'])),
               'decode': cfg['decode'], 'n_train': len(rows),
               'n_features': x_fit.shape[1],
               'fit_sec': round(time.time() - start, 1),
               'finished': pd.Timestamp.now().isoformat(timespec='seconds')}
        pd.concat([results, pd.DataFrame([row])]).to_csv(RESULTS, index=False)
        print(dataset, cfg['exp_id'], round(row['macro_f1'], 4), flush=True)
    return table(group)

## 1. Features
Base model: logreg C=2 with balanced class weights.

In [8]:
run('features', [
    # Words only.
    exp('f01_word11', feat(word=[1, 1])),
    # Words and bigrams, the baseline.
    exp('f02_word12', feat()),
    # Up to trigrams with a 300k vocabulary.
    exp('f03_word13', feat(word=[1, 3], max_features=300_000)),
    # Full vocabulary with min_df=2.
    exp('f04_word12_full_vocab', feat(min_df=2, max_features=None)),
    # Vocabulary cut to 50k.
    exp('f05_word12_50k', feat(max_features=50_000)),
    # Raw term frequency instead of sublinear.
    exp('f06_word12_raw_tf', feat(sublinear=False)),
    # Binary term frequency.
    exp('f07_word12_binary', feat(binary=True)),
    # Character 2-5 grams only.
    exp('f08_char25', feat(word=None, char=[2, 5])),
    # Character 1-4 grams only.
    exp('f09_char14', feat(word=None, char=[1, 4])),
    # Words, bigrams and character 2-5 grams.
    exp('f10_word12_char25', feat(char=[2, 5])),
    # Text without the pros/cons field markers.
    exp('f11_plain', feat(text='plain')),
    # No markers, field flags as explicit features.
    exp('f12_plain_flags', feat(text='plain', meta=['flags'])),
    # Field flags, length and punctuation.
    exp('f13_meta', feat(meta=['flags', 'length', 'punct'])),
    # Field flags, length, punctuation and product category.
    exp('f14_meta_cat', feat(meta=ALL_META)),
    # Normalized text: emoji, exclamations, digits, ё.
    exp('f15_clean', feat(text='clean')),
    # Snowball stems.
    exp('f16_stem', feat(text='stem')),
    # Pymorphy lemmas, Russian only.
    exp('f17_lemma', feat(text='lemma'), wb_only=True),
    # Stems, character 2-5 grams and all meta features.
    exp('f18_stem_char_meta',
        feat(text='stem', char=[2, 5], meta=ALL_META)),
])

macro_f1             mae         fit_sec       
dataset                 amazon      wb  amazon      wb  amazon     wb
exp_id                                                               
f13_meta                0.5253  0.4614  0.3645  0.3671    89.1   42.9
f15_clean               0.5235  0.4604  0.3674  0.3673    40.3   19.9
f16_stem                0.5166  0.4603  0.3713  0.3663    46.4   17.6
f04_word12_full_vocab   0.5255  0.4602  0.3627  0.3638    74.4   35.6
f14_meta_cat            0.5261  0.4598  0.3652  0.3698    95.0   45.1
f10_word12_char25       0.5212  0.4595  0.3709  0.3609   281.0  174.1
f17_lemma                  NaN  0.4592     NaN  0.3666     NaN   21.0
f18_stem_char_meta      0.5156  0.4589  0.3745  0.3661   332.7  193.7
f12_plain_flags         0.5234  0.4584  0.3672  0.3721    88.8   34.5
f03_word13              0.5261  0.4581  0.3586  0.3670    58.8   25.0
f06_word12_raw_tf       0.5224  0.4580  0.3708  0.3720    39.4   23.0
f02_word12              0.5224  0.4574  0.3691  0.3713    36.6   25.3
f07_word12_binary       0.5206  0.4571  0.3707  0.3710    38.9   22.8
f11_plain               0.5224  0.4515  0.3691  0.3933    40.3   21.4
f05_word12_50k          0.5117  0.4471  0.3902  0.3915    28.9   14.4
f08_char25              0.5006  0.4431  0.4342  0.4161   190.2  157.4
f09_char14              0.4970  0.4414  0.4421  0.4191   155.0  103.3
f01_word11              0.4920  0.4322  0.4493  0.4440    17.0   13.4

Bigrams are the only large gain: +0.025 on WB and +0.030 on Amazon over single words. No other change adds more than 0.004, and a 50k vocabulary loses 0.010. Character n-grams alone lose 0.014–0.025 and fit 4–6 times slower.

Best: WB `f13_meta` (0.4614), Amazon `f03_word13` (0.5261). Later groups start from these.

## 2. Models
Best features of each dataset, balanced class weights.

In [9]:
run('models', [
    # Logistic regression, inverse regularization grid.
    *sweep('logreg', 'C', [0.5, 1.0, 4.0, 8.0]),
    # Linear SVM, inverse regularization grid.
    *sweep('linsvc', 'C', [0.03, 0.1, 0.3, 1.0]),
    # SGD with modified Huber loss.
    *sweep('sgd', 'alpha', [1e-5, 3e-6], loss='modified_huber'),
    # SGD with log loss.
    *sweep('sgd', 'alpha', [3e-6], loss='log_loss'),
    # SGD with hinge loss.
    *sweep('sgd', 'alpha', [1e-5], loss='hinge'),
    # Ridge classifier.
    *sweep('ridge_clf', 'alpha', [1.0, 3.0, 10.0]),
    # Complement naive Bayes.
    *sweep('cnb', 'alpha', [0.1, 0.3, 1.0]),
    # Multinomial naive Bayes.
    *sweep('mnb', 'alpha', [0.03, 0.1]),
    # Logistic regression on naive Bayes ratios (NB-SVM).
    *sweep('nbsvm', 'C', [1.0, 4.0]),
    # Passive-aggressive via SGD, which accepts sample weights.
    *sweep('pa', 'C', [0.01, 0.1]),
])

macro_f1             mae         fit_sec  \
dataset                               amazon      wb  amazon      wb  amazon   
exp_id                                                                         
m_linsvc_C0.03                        0.5298  0.4683  0.3659  0.3605     3.6   
m_nbsvm_C1.0                          0.5278  0.4664  0.3686  0.3467    16.7   
m_ridge_clf_alpha10.0                 0.5296  0.4636  0.3587  0.3588     3.1   
m_logreg_C1.0                         0.5297  0.4634  0.3663  0.3704    33.9   
m_logreg_C0.5                         0.5294  0.4634  0.3775  0.3752    30.8   
m_linsvc_C0.1                         0.5304  0.4613  0.3544  0.3628     4.9   
m_nbsvm_C4.0                          0.5199  0.4576  0.3611  0.3555    24.2   
m_linsvc_C0.3                         0.5200  0.4567  0.3583  0.3676     9.4   
m_ridge_clf_alpha3.0                  0.5223  0.4563  0.3591  0.3702     5.2   
m_logreg_C4.0                         0.5214  0.4541  0.3542  0.3664    66.2   
m_sgd_losshinge_alpha1e-05            0.5201  0.4532  0.3635  0.3721     2.8   
m_sgd_losslog_loss_alpha3e-06         0.5290  0.4501  0.3585  0.3892     2.8   
m_logreg_C8.0                         0.5169  0.4492  0.3539  0.3681    67.8   
m_mnb_alpha0.1                        0.5116  0.4481  0.3952  0.4031     0.1   
m_mnb_alpha0.03                       0.5071  0.4444  0.3816  0.3953     0.1   
m_sgd_lossmodified_huber_alpha1e-05   0.5105  0.4438  0.3672  0.3734     2.3   
m_ridge_clf_alpha1.0                  0.5047  0.4436  0.3736  0.3915     8.9   
m_cnb_alpha1.0                        0.5092  0.4420  0.4088  0.4043     0.1   
m_linsvc_C1.0                         0.5037  0.4408  0.3687  0.3824    22.1   
m_cnb_alpha0.3                        0.5033  0.4336  0.4075  0.4145     0.1   
m_sgd_lossmodified_huber_alpha3e-06   0.4922  0.4277  0.3824  0.3936     2.0   
m_cnb_alpha0.1                        0.4991  0.4267  0.4084  0.4214     0.1   
m_pa_C0.1                             0.4740  0.3987  0.3914  0.4145     5.0   
m_pa_C0.01                            0.4740  0.3987  0.3914  0.4145     4.6   

                                            
dataset                                 wb  
exp_id                                      
m_linsvc_C0.03                         7.7  
m_nbsvm_C1.0                          13.6  
m_ridge_clf_alpha10.0                  2.7  
m_logreg_C1.0                         33.3  
m_logreg_C0.5                         26.5  
m_linsvc_C0.1                        263.5  
m_nbsvm_C4.0                          20.1  
m_linsvc_C0.3                        272.1  
m_ridge_clf_alpha3.0                   4.5  
m_logreg_C4.0                         54.6  
m_sgd_losshinge_alpha1e-05             2.2  
m_sgd_losslog_loss_alpha3e-06          2.7  
m_logreg_C8.0                         54.3  
m_mnb_alpha0.1                         0.1  
m_mnb_alpha0.03                        0.1  
m_sgd_lossmodified_huber_alpha1e-05    2.7  
m_ridge_clf_alpha1.0                   7.5  
m_cnb_alpha1.0                         0.1  
m_linsvc_C1.0                         44.9  
m_cnb_alpha0.3                         0.1  
m_sgd_lossmodified_huber_alpha3e-06    3.2  
m_cnb_alpha0.1                         0.1  
m_pa_C0.1                              3.7  
m_pa_C0.01                             3.6

Strong regularization wins: logreg C=0.5–1, LinearSVC C=0.03 on WB and 0.1 on Amazon. LinearSVC is the best family here (0.4683 / 0.5304). Logreg, ridge (alpha=10) and NB-SVM (C=1) are within 0.005 on both datasets.

Naive Bayes reaches at most 0.448 / 0.512. Passive-aggressive is the weakest (0.399 / 0.474), and its two C values give identical results.

## 3. Class imbalance
Best features, logreg C=2.

In [10]:
run('imbalance', [
    # No reweighting.
    exp('i_none', imbalance='none'),
    # Square root of balanced weights.
    exp('i_sqrt', imbalance='sqrt'),
    # Undersample every class to the rarest one.
    exp('i_under', imbalance='under'),
    # Undersample only 5★, to twice the next class.
    exp('i_under5', imbalance='under5'),
    # Oversample minority classes to half of 5★.
    exp('i_over', imbalance='over'),
])

macro_f1             mae         fit_sec      
dataset    amazon      wb  amazon      wb  amazon    wb
exp_id                                                 
i_sqrt     0.5327  0.4685  0.3099  0.3098    54.6  35.0
i_over     0.5242  0.4596  0.3234  0.3342    98.3  73.7
i_under5   0.5208  0.4554  0.3231  0.3321    38.6  20.5
i_none     0.5034  0.4376  0.3168  0.3136    45.1  23.5
i_under    0.5047  0.4368  0.4343  0.4367    19.7  11.5

Square-root weights are the best trade-off: macro-F1 0.4685 / 0.5327 and MAE 0.310 on both datasets. With no weights, 2★ almost disappears on WB (F1 0.047). Full undersampling has the worst MAE (0.437 / 0.434), and its macro-F1 (0.437 / 0.505) is about as low as with no weights.

## 4. Train data cleaning
Validation is never filtered.

In [11]:
run('data', [
    # Drop train reviews shorter than 3 words.
    exp('d_drop_short', filters=['drop_short']),
    # Drop duplicate texts in train.
    exp('d_dedup', filters=['dedup']),
    # Drop both short reviews and duplicates.
    exp('d_drop_short_dedup', filters=['drop_short', 'dedup']),
    # Drop reviews about delivery, WB only.
    exp('d_drop_not_product', filters=['drop_not_product'], wb_only=True),
])

macro_f1             mae         fit_sec      
dataset              amazon      wb  amazon      wb  amazon    wb
exp_id                                                           
d_drop_short_dedup   0.5279  0.4631  0.3561  0.3635    53.9  40.0
d_dedup              0.5277  0.4631  0.3576  0.3644    54.7  37.3
d_drop_short         0.5288  0.4627  0.3568  0.3639    65.9  39.4
d_drop_not_product      NaN  0.4598     NaN  0.3681     NaN  42.5

Gains of at most 0.002–0.003 are too small to rely on. Dropping delivery reviews lowers WB macro-F1 by 0.002.

## 5. Decision rules
Offsets and thresholds are tuned on out-of-fold scores of the fit part. Validation is not used for tuning.

In [12]:
run('decoding', [
    # Median of the predicted distribution.
    exp('o_logreg_median', decode='median'),
    # Balanced weights with per-class offsets.
    exp('o_logreg_bias', decode='bias'),
    # No weights with per-class offsets.
    exp('o_logreg_noweight_bias', imbalance='none', decode='bias'),
    # Linear SVM with per-class offsets.
    exp('o_linsvc_bias', model={'name': 'linsvc', 'C': 0.1}, decode='bias'),
    # Ridge regression, rounded.
    exp('o_ridge_reg_round', model={'name': 'ridge_reg', 'alpha': 1.0}),
    # Ridge regression alpha=1 with tuned thresholds.
    exp('o_ridge_reg_thr1', model={'name': 'ridge_reg', 'alpha': 1.0},
        decode='thresholds'),
    # Ridge regression alpha=3 with tuned thresholds.
    exp('o_ridge_reg_thr3', model={'name': 'ridge_reg', 'alpha': 3.0},
        decode='thresholds'),
])

macro_f1             mae         fit_sec       
dataset                  amazon      wb  amazon      wb  amazon     wb
exp_id                                                                
o_logreg_noweight_bias   0.5397  0.4784  0.3156  0.3054   156.1  108.2
o_logreg_bias            0.5333  0.4692  0.3255  0.3303   189.7  166.6
o_linsvc_bias            0.5350  0.4685  0.3358  0.3262    36.6   51.8
o_logreg_median          0.5227  0.4531  0.3769  0.3783    60.7   46.1
o_ridge_reg_thr3         0.4836  0.4410  0.3529  0.3371    17.2   15.6
o_ridge_reg_thr1         0.4762  0.4274  0.3590  0.3553    18.7   17.4
o_ridge_reg_round        0.4225  0.3628  0.5579  0.5565     2.1    1.4

The largest single gain: no weights plus offsets adds 0.017 / 0.014 over the best features (0.4784 / 0.5397). Offsets on top of balanced weights add only about 0.008.

The median does not help with balanced weights (MAE 0.378 / 0.377). Ridge regression is worse than classification, even with tuned thresholds (at most 0.441 / 0.484).

## 6. Feature selection
Chi-squared selection on the best features.

In [13]:
run('selection', [
    # Keep the top 20k, 50k or 100k features.
    exp(f's_chi2_{k // 1000}k', {**BEST, 'select_k': k})
    for k in [20_000, 50_000, 100_000]
])

macro_f1             mae         fit_sec      
dataset       amazon      wb  amazon      wb  amazon    wb
exp_id                                                    
s_chi2_100k   0.5227  0.4615  0.3722  0.3672    26.1  37.6
s_chi2_50k    0.5200  0.4582  0.3834  0.3755    18.6  30.2
s_chi2_20k    0.5205  0.4534  0.3939  0.3860    12.5  19.5

No gain: 100k features match the full set on WB and lose 0.003 on Amazon. Smaller sets are worse.

## 7. Combinations
The strongest findings together.

In [14]:
LOGREG_1 = {'name': 'logreg', 'C': 1.0}
SVC_03 = {'name': 'linsvc', 'C': 0.03}
WITH_CHAR = {**BEST, 'char': [2, 5]}

run('combo', [
    # Logreg C=1, no weights, offsets.
    exp('c_logreg1_none_bias', model=LOGREG_1, imbalance='none', decode='bias'),
    # Logreg C=1, square-root weights, offsets.
    exp('c_logreg1_sqrt_bias', model=LOGREG_1, imbalance='sqrt', decode='bias'),
    # Logreg C=1, no weights, median for MAE.
    exp('c_logreg1_none_median', model=LOGREG_1, imbalance='none',
        decode='median'),
    # Linear SVM, no weights, offsets.
    exp('c_svc_none_bias', model=SVC_03, imbalance='none', decode='bias'),
    # Linear SVM, square-root weights, offsets.
    exp('c_svc_sqrt_bias', model=SVC_03, imbalance='sqrt', decode='bias'),
    # NB-SVM, no weights, offsets.
    exp('c_nbsvm_none_bias', model={'name': 'nbsvm', 'C': 1.0},
        imbalance='none', decode='bias'),
    # Ridge classifier, no weights, offsets.
    exp('c_ridge_none_bias', model={'name': 'ridge_clf', 'alpha': 10.0},
        imbalance='none', decode='bias'),
    # Best logreg trained without short reviews and duplicates.
    exp('c_logreg1_none_bias_clean', model=LOGREG_1, imbalance='none',
        filters=['drop_short', 'dedup'], decode='bias'),
    # Best logreg with character n-grams.
    exp('c_logreg1_none_bias_char', WITH_CHAR, LOGREG_1, 'none', decode='bias'),
    # Linear SVM with character n-grams.
    exp('c_svc_none_bias_char', WITH_CHAR, SVC_03, 'none', decode='bias'),
])

macro_f1             mae         fit_sec       
dataset                     amazon      wb  amazon      wb  amazon     wb
exp_id                                                                   
c_logreg1_none_bias         0.5437  0.4838  0.3158  0.3057   115.7   86.2
c_logreg1_none_bias_clean   0.5436  0.4835  0.3181  0.3051   124.3   83.5
c_logreg1_none_bias_char    0.5437  0.4833  0.3158  0.3007   513.8  418.2
c_nbsvm_none_bias           0.5364  0.4795  0.3108  0.3033    57.1   46.2
c_logreg1_sqrt_bias         0.5415  0.4779  0.3209  0.3077   140.4  107.9
c_svc_none_bias_char        0.5386  0.4770  0.3152  0.3118    57.5   43.0
c_svc_sqrt_bias             0.5393  0.4748  0.3269  0.3281    28.7   35.5
c_ridge_none_bias           0.5339  0.4706  0.3287  0.3318    27.8   24.9
c_logreg1_none_median       0.5323  0.4693  0.2941  0.2890    31.5   20.3
c_svc_none_bias             0.5317  0.4692  0.3354  0.3335    27.8   36.0

Logreg C=1 without weights plus offsets is the best on both datasets. NB-SVM follows (0.4795 / 0.5364) and is a candidate for an ensemble. Character n-grams do not help logreg but add 0.008 / 0.007 to the linear SVM, which stays below logreg.

## 8. Pretrained word2vec
Word vectors averaged per review, alone and next to TF-IDF. WB: RusCorpora lemmas (about 63% of tokens are covered, the model has no function words such as «не»). Amazon: Google News (about 90% covered). IDF weights are fitted on the fit part.

In [15]:
MEAN = feat(word=None, vectors='mean')
IDF = feat(word=None, vectors='idf')
WITH_MEAN = {**BEST, 'vectors': 'mean'}
WITH_IDF = {**BEST, 'vectors': 'idf'}
NONE_BIAS = dict(imbalance='none', decode='bias')

run('word2vec', [
    # Mean vectors only.
    exp('w_mean', MEAN),
    # IDF-weighted mean vectors only.
    exp('w_idf', IDF),
    # Mean vectors, weaker regularization.
    exp('w_mean_c10', MEAN, {'name': 'logreg', 'C': 10.0}),
    # Mean vectors, almost no regularization.
    exp('w_mean_c100', MEAN, {'name': 'logreg', 'C': 100.0}),
    # Mean vectors with a linear SVM.
    exp('w_mean_svc', MEAN, {'name': 'linsvc', 'C': 1.0}),
    # IDF-weighted vectors, no weights, offsets.
    exp('w_idf_none_bias', IDF, {'name': 'logreg', 'C': 10.0}, **NONE_BIAS),
    # Best TF-IDF plus mean vectors.
    exp('w_tfidf_mean', WITH_MEAN),
    # Best TF-IDF plus IDF-weighted vectors.
    exp('w_tfidf_idf', WITH_IDF),
    # Best TF-IDF plus mean vectors scaled by 3.
    exp('w_tfidf_mean_x3', {**WITH_MEAN, 'vector_scale': 3.0}),
    # Best rule on TF-IDF plus mean vectors.
    exp('w_best_mean_bias', WITH_MEAN, LOGREG_1, **NONE_BIAS),
    # Best rule on TF-IDF plus IDF-weighted vectors.
    exp('w_best_idf_bias', WITH_IDF, LOGREG_1, **NONE_BIAS),
    # Best rule on TF-IDF plus mean vectors scaled by 3.
    exp('w_best_mean_x3_bias', {**WITH_MEAN, 'vector_scale': 3.0}, LOGREG_1,
        **NONE_BIAS),
    # TF-IDF plus mean vectors, median for MAE.
    exp('w_best_mean_median', WITH_MEAN, LOGREG_1, 'none', decode='median'),
])

macro_f1             mae         fit_sec       
dataset               amazon      wb  amazon      wb  amazon     wb
exp_id                                                             
w_best_idf_bias       0.5442  0.4846  0.3142  0.3065   552.6  610.4
w_best_mean_x3_bias   0.5422  0.4830  0.3159  0.3092   901.1  604.7
w_best_mean_bias      0.5437  0.4816  0.3177  0.3089   576.0  540.3
w_best_mean_median    0.5343  0.4755  0.2931  0.2851   128.9  133.0
w_tfidf_mean          0.5289  0.4608  0.3609  0.3680   206.9  245.6
w_tfidf_mean_x3       0.5269  0.4600  0.3615  0.3693   592.4  666.2
w_tfidf_idf           0.5256  0.4600  0.3614  0.3694   210.5  341.7
w_mean_svc            0.4692  0.4015  0.5259  0.5474    48.9   36.0
w_idf_none_bias       0.4603  0.3962  0.4861  0.4666   167.1  110.2
w_mean_c100           0.4629  0.3957  0.5467  0.5785    88.7   47.2
w_mean                0.4642  0.3951  0.5477  0.5784    76.3   44.6
w_mean_c10            0.4634  0.3951  0.5457  0.5787    73.4   38.8
w_idf                 0.4404  0.3744  0.6109  0.6539    53.0   32.4

Vectors alone are far weaker than TF-IDF: at best 0.4015 / 0.4692 with MAE 0.547 / 0.526.

On top of TF-IDF with the best rule, IDF-weighted vectors give 0.4846 / 0.5442 against 0.4838 / 0.5437 without them. That gain is too small to rely on and costs 5–7 times more fit time. Scaling the vectors by 3 does not help.

The useful effect is on MAE with the median rule: 0.2851 / 0.2931 against 0.2890 / 0.2941, with macro-F1 up by 0.006 / 0.002.

## 9. Optuna
TPE search without word2vec. Each trial fits on 75% of the fit part. Offsets are tuned on one half of the remaining 25% and macro-F1 is measured on the other half, split by product. Validation scores only the best config of each dataset.

In [16]:
OPTUNA_TRIALS = 40
OPTUNA_DB = 'sqlite:///optuna.db'
C_RANGES = {'logreg': (0.1, 10.0), 'linsvc': (0.005, 1.0), 'nbsvm': (0.1, 10.0)}
OPTUNA_META = ['flags', 'length', 'punct']


def suggest(trial):
    """Samples one pipeline configuration."""
    model = trial.suggest_categorical('model', list(C_RANGES))
    trial.suggest_float(f'C_{model}', *C_RANGES[model], log=True)
    trial.suggest_categorical('text', ['full', 'clean'])
    trial.suggest_int('word_max', 2, 3)
    trial.suggest_int('min_df', 2, 10)
    trial.suggest_int('max_features', 50_000, 500_000, log=True)
    trial.suggest_categorical('sublinear', [True, False])
    trial.suggest_categorical('meta', [True, False])
    trial.suggest_float('weight_power', 0.0, 1.0)
    return trial.params


def specs_from(params):
    """Returns the feature and model specs of a sampled configuration."""
    features = feat(text=params['text'], word=[1, params['word_max']],
                    min_df=params['min_df'],
                    max_features=params['max_features'],
                    sublinear=params['sublinear'],
                    meta=OPTUNA_META if params['meta'] else [])
    name = params['model']
    return features, {'name': name, 'C': params[f'C_{name}']}


def inner_score(dataset, params):
    """Returns macro-F1 on the inner hold-out, with tuned offsets."""
    data = DATA[dataset]
    features, spec = specs_from(params)
    key = json.dumps(features, sort_keys=True)
    x_tr, x_te = build_features(dataset, key, 'inner')
    tr, te = data.splits['inner']
    y = data.df.rating.to_numpy()
    weight = class_weights(y[tr], params['weight_power'])
    s = scores(make_model(spec).fit(x_tr, y[tr], sample_weight=weight), x_te)
    splitter = model_selection.GroupShuffleSplit(
        n_splits=1, test_size=0.5, random_state=SEED)
    groups = data.df.product_id.to_numpy()[te]
    tune_part, check = next(splitter.split(te, groups=groups))
    offsets = tune_bias(s[tune_part], y[te][tune_part])
    pred = CLASSES[(s[check] + offsets).argmax(1)]
    return metrics.f1_score(y[te][check], pred, average='macro')


def tune(dataset):
    """Runs the remaining trials of a persistent study."""
    study = optuna.create_study(
        study_name=f'part_a_{dataset}', storage=OPTUNA_DB,
        direction='maximize', load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda t: inner_score(dataset, suggest(t)),
                   n_trials=OPTUNA_TRIALS - len(study.trials))
    return study


optuna.logging.set_verbosity(optuna.logging.WARNING)
studies = {dataset: tune(dataset) for dataset in DATASETS}
for dataset, study in studies.items():
    features, model = specs_from(study.best_params)
    # The best config is scored once on validation.
    run('optuna', [exp('opt_best_bias', features, model,
                       study.best_params['weight_power'], decode='bias')],
        [dataset])
table('optuna')

macro_f1             mae         fit_sec      
dataset         amazon      wb  amazon      wb  amazon    wb
exp_id                                                      
opt_best_bias   0.5418  0.4752  0.3113  0.3085    47.0  29.4

In [17]:
# Best configuration and inner hold-out macro-F1 of each study.
pd.DataFrame({dataset: {'inner_macro_f1': round(study.best_value, 4),
                        **study.best_params}
              for dataset, study in studies.items()})

,wb,amazon
inner_macro_f1,0.469,0.5441
model,nbsvm,nbsvm
C_nbsvm,0.318729,1.033367
text,clean,clean
word_max,3,2
min_df,9,9
max_features,189628,366055
sublinear,True,False
meta,True,True
weight_power,0.011548,0.094185


Both searches converged to NB-SVM with almost no class weights (power 0.01 / 0.09), `min_df=9`, normalized text and meta features.

Validation macro-F1 is 0.4752 / 0.5418, below the hand-built best (0.4838 / 0.5437) by 0.009 / 0.002. Its config was chosen without validation, while the hand-built one was chosen on it, so the real gap may be smaller. Validation MAE is 0.3085 / 0.3113. Forty trials found no improvement on macro-F1.

## 10. Text only
Meta features belong to part B, so the final model uses text alone. The best text-only spec of the features group replaces the best overall one. On Amazon it is the same spec.

In [18]:
TEXT_BEST = {'from': 'best_text'}

run('text_only', [
    # Best rule on the best text-only features.
    exp('t_logreg1_none_bias', TEXT_BEST, LOGREG_1, **NONE_BIAS),
    # Same model, median for MAE.
    exp('t_logreg1_none_median', TEXT_BEST, LOGREG_1, 'none',
        decode='median'),
])

macro_f1             mae         fit_sec      
dataset                 amazon      wb  amazon      wb  amazon    wb
exp_id                                                              
t_logreg1_none_bias     0.5437  0.4822  0.3158  0.3079   146.8  61.7
t_logreg1_none_median   0.5323  0.4666  0.2941  0.2866    42.3  13.8

WB picks normalized text (`f15_clean`). Dropping meta features costs 0.0016 macro-F1 (0.4822 against 0.4838), which is within noise, and the median MAE even improves (0.2866 against 0.2890). Amazon is unchanged because its best spec had no meta features.

In [19]:
SVC_RAW = {'name': 'linsvc', 'C': 0.03}
SVC_CAL = {method: {'name': 'linsvc_cal', 'C': 0.03, 'method': method}
           for method in ['sigmoid', 'isotonic']}

run('text_only', [
    # Linear SVM on raw margins with offsets.
    exp('t_svc_none_bias', TEXT_BEST, SVC_RAW, **NONE_BIAS),
    # Platt-calibrated SVM with offsets.
    exp('t_svc_sigmoid_bias', TEXT_BEST, SVC_CAL['sigmoid'], **NONE_BIAS),
    # Platt-calibrated SVM, median for MAE.
    exp('t_svc_sigmoid_median', TEXT_BEST, SVC_CAL['sigmoid'], 'none',
        decode='median'),
    # Isotonic-calibrated SVM with offsets.
    exp('t_svc_isotonic_bias', TEXT_BEST, SVC_CAL['isotonic'], **NONE_BIAS),
    # Isotonic-calibrated SVM, median for MAE.
    exp('t_svc_isotonic_median', TEXT_BEST, SVC_CAL['isotonic'], 'none',
        decode='median'),
])

wb t_svc_none_bias 0.4689


wb t_svc_sigmoid_bias 0.4729


wb t_svc_sigmoid_median 0.4559


wb t_svc_isotonic_bias 0.4692


wb t_svc_isotonic_median 0.4592


amazon t_svc_none_bias 0.5317


amazon t_svc_sigmoid_bias 0.5308


amazon t_svc_sigmoid_median 0.5126


amazon t_svc_isotonic_bias 0.5317


amazon t_svc_isotonic_median 0.5213


macro_f1             mae         fit_sec      
dataset                 amazon      wb  amazon      wb  amazon    wb
exp_id                                                              
t_logreg1_none_bias     0.5437  0.4822  0.3158  0.3079   146.8  61.7
t_svc_sigmoid_bias      0.5308  0.4729  0.3400  0.3248    36.2  24.2
t_svc_isotonic_bias     0.5317  0.4692  0.3378  0.3264    35.7  25.6
t_svc_none_bias         0.5317  0.4689  0.3354  0.3355    26.1  22.1
t_logreg1_none_median   0.5323  0.4666  0.2941  0.2866    42.3  13.8
t_svc_isotonic_median   0.5213  0.4592  0.3131  0.3010     6.0   3.9
t_svc_sigmoid_median    0.5126  0.4559  0.3249  0.3111     5.7   3.5

Calibration does not make the SVM competitive. With offsets it reaches at most 0.4729 / 0.5317 against 0.4822 / 0.5437 for logreg. With the median, MAE is 0.301 / 0.313 against 0.287 / 0.294.

## 11. Summary on validation

In [20]:
# Best macro-F1 reached in each group.
results = pd.read_csv(RESULTS)
results.pivot_table(index='group', columns='dataset', values='macro_f1',
                    aggfunc='max').sort_values('wb', ascending=False).round(4)

dataset,amazon,wb
group,,
word2vec,0.5442,0.4846
combo,0.5437,0.4838
text_only,0.5437,0.4822
decoding,0.5397,0.4784
optuna,0.5418,0.4752
imbalance,0.5327,0.4685
models,0.5304,0.4683
data,0.5288,0.4631
selection,0.5227,0.4615


In [21]:
# Baseline against the best runs, with per-class F1.
key_runs = ['f02_word12', 'c_logreg1_none_bias', 'c_logreg1_none_median',
            'w_best_idf_bias', 'w_best_mean_median', 'opt_best_bias',
            't_logreg1_none_bias', 't_logreg1_none_median']
columns = ['macro_f1', 'mae', 'accuracy', *[f'f1_{c}' for c in CLASSES]]
(results[results.exp_id.isin(key_runs)]
 .set_index(['dataset', 'exp_id'])[columns].sort_index().round(3))

macro_f1    mae  accuracy   f1_1   f1_2   f1_3  \
dataset exp_id                                                                  
amazon  c_logreg1_none_bias       0.544  0.316     0.761  0.702  0.321  0.393   
        c_logreg1_none_median     0.532  0.294     0.777  0.709  0.285  0.389   
        f02_word12                0.522  0.369     0.723  0.701  0.284  0.377   
        opt_best_bias             0.542  0.311     0.767  0.720  0.287  0.384   
        t_logreg1_none_bias       0.544  0.316     0.761  0.702  0.321  0.393   
        t_logreg1_none_median     0.532  0.294     0.777  0.709  0.285  0.389   
        w_best_idf_bias           0.544  0.314     0.762  0.701  0.323  0.392   
        w_best_mean_median        0.534  0.293     0.777  0.718  0.292  0.385   
wb      c_logreg1_none_bias       0.484  0.306     0.797  0.566  0.191  0.328   
        c_logreg1_none_median     0.469  0.289     0.809  0.517  0.181  0.340   
        f02_word12                0.457  0.371     0.762  0.550  0.162  0.292   
        opt_best_bias             0.475  0.308     0.800  0.587  0.143  0.303   
        t_logreg1_none_bias       0.482  0.308     0.798  0.581  0.168  0.332   
        t_logreg1_none_median     0.467  0.287     0.809  0.509  0.171  0.344   
        w_best_idf_bias           0.485  0.307     0.799  0.579  0.193  0.323   
        w_best_mean_median        0.476  0.285     0.810  0.543  0.184  0.344   

                                f1_4   f1_5  
dataset exp_id                               
amazon  c_logreg1_none_bias    0.395  0.908  
        c_logreg1_none_median  0.357  0.921  
        f02_word12             0.373  0.877  
        opt_best_bias          0.404  0.913  
        t_logreg1_none_bias    0.395  0.908  
        t_logreg1_none_median  0.357  0.921  
        w_best_idf_bias        0.396  0.909  
        w_best_mean_median     0.356  0.920  
wb      c_logreg1_none_bias    0.398  0.936  
        c_logreg1_none_median  0.366  0.942  
        f02_word12             0.366  0.916  
        opt_best_bias          0.406  0.937  
        t_logreg1_none_bias    0.394  0.936  
        t_logreg1_none_median  0.365  0.943  
        w_best_idf_bias        0.391  0.937  
        w_best_mean_median     0.364  0.942

- **Final model:** text only, logreg C=1 without weights. Offsets are used for macro-F1 and the median for MAE.
- **Results** (validation macro-F1 / MAE):
  - WB: 0.457 / 0.371 → 0.482 / 0.308, median MAE 0.287.
  - Amazon: 0.522 / 0.369 → 0.544 / 0.316, median MAE 0.294.
- **With meta features on WB** (part B territory): 0.484 / 0.306, median MAE 0.289.
- **With pretrained word2vec:** macro-F1 0.485 / 0.544, best MAE 0.285 / 0.293. Only the MAE gain is meaningful. The WB runs include meta features.
- **Optuna:** 0.475 / 0.542 on validation, no improvement. It also chose meta features.
- **Main driver:** the decision rule, about +0.02, not the features or the model family.
- **One decoder per metric:** offsets for macro-F1, median for MAE, on the same unweighted model.
- **Hardest classes:** 2★ and 3★, with F1 0.19 / 0.33 on WB and 0.32 / 0.39 on Amazon.
- **Caveat:** configurations were chosen on the validation split, so these scores are slightly optimistic. Differences below about 0.003 should not be over-read. The final estimate is on test below.

## 12. Test
The final text-only model is fitted on the whole train split. Offsets are tuned on 3 folds of train by product. The test split is scored once. The baseline is shown for reference only and was not used to choose anything.

In [22]:
def test_scores(dataset, cfg, results):
    """Fits on the whole train split, returns metrics on the test split."""
    data = DATA[dataset]
    spec = resolve_features(results, dataset, cfg['features'])
    key = json.dumps(spec, sort_keys=True)
    x_tr, x_te = build_features(dataset, key, 'test')
    tr, te = data.splits['test']
    y = data.df.rating.to_numpy()
    weight = class_weights(y[tr], WEIGHT_POWER[cfg['imbalance']])
    groups = data.df.product_id.to_numpy()[tr]
    pred = predict(cfg, x_tr, y[tr], weight, groups, x_te)
    return {'n_test': len(te), **evaluate(y[te], pred)}


FINAL = {
    # TF-IDF words and bigrams, logreg C=2, balanced weights.
    'baseline': exp('f02_word12', feat()),
    # Final model with offsets, for macro-F1.
    'final, offsets': exp('t_logreg1_none_bias', TEXT_BEST, LOGREG_1,
                          **NONE_BIAS),
    # Final model with the median, for MAE.
    'final, median': exp('t_logreg1_none_median', TEXT_BEST, LOGREG_1, 'none',
                         decode='median'),
}
results = pd.read_csv(RESULTS)
pd.DataFrame({(dataset, name): test_scores(dataset, cfg, results)
              for dataset in DATASETS
              for name, cfg in FINAL.items()}).T.round(3)

n_test  macro_f1    mae  accuracy   f1_1   f1_2  \
wb     baseline        58127.0     0.457  0.368     0.764  0.549  0.164   
       final, offsets  58127.0     0.484  0.301     0.801  0.591  0.169   
       final, median   58127.0     0.470  0.281     0.812  0.531  0.173   
amazon baseline        60736.0     0.520  0.382     0.711  0.708  0.286   
       final, offsets  60736.0     0.546  0.325     0.755  0.712  0.330   
       final, median   60736.0     0.532  0.311     0.765  0.710  0.291   

                        f1_3   f1_4   f1_5  
wb     baseline        0.290  0.362  0.918  
       final, offsets  0.329  0.394  0.938  
       final, median   0.337  0.367  0.944  
amazon baseline        0.357  0.376  0.873  
       final, offsets  0.383  0.397  0.907  
       final, median   0.377  0.365  0.917

Test agrees with validation. With offsets: WB 0.484 / 0.301 (validation 0.482 / 0.308), Amazon 0.546 / 0.325 (validation 0.544 / 0.316). With the median, MAE is 0.281 / 0.311.

Against the baseline on test: +0.027 / +0.026 macro-F1 with offsets and −0.087 / −0.071 MAE with the median. 2★ stays the hardest class (F1 0.17 / 0.33).